In [ ]:
# Import the libraries required for environment variables, website retrieval, and the OpenAI client.
import os

from dotenv import load_dotenv
from scrape import fetch_website_contents
from openai import OpenAI

In [ ]:
# Load the OpenAI API key and initialize the OpenAI client.
load_dotenv(override=True)

api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    print("No API key was found, kindly recheck this.")
else:
    print("API key found.")

openai = OpenAI()

In [ ]:
# Define the research question and the websites that will provide the research material.
research_question = """
What are the major applications of AI automation in modern businesses?
"""

research_urls = [
    "https://www.ibm.com/think/topics/ai-automation",
    "https://www.microsoft.com/en-us/ai",
    "https://www.salesforce.com/artificial-intelligence/"
]

In [ ]:
# Retrieve and combine the content from all research sources.
def collect_sources(urls):
    sources = []

    for url in urls:
        try:
            content = fetch_website_contents(url)
            sources.append({
                "url": url,
                "content": content
            })
        except Exception as error:
            print(f"Could not retrieve {url}: {error}")

    return sources


sources = collect_sources(research_urls)

print(f"Sources collected: {len(sources)}")

In [ ]:
# Define the instructions for synthesizing information from multiple research sources.
research_prompt = """
You are a research assistant.

Your task is to answer the research question using the provided sources.

Requirements:

1. Identify the major findings relevant to the research question.
2. Combine information from multiple sources where appropriate.
3. Do not invent information that is not supported by the sources.
4. Distinguish information supported by the sources from conclusions or interpretations.
5. Identify important similarities or differences between sources.
6. Mention which source supports important findings.
7. Ignore navigation text, advertisements, and unrelated website content.

Return the research in concise Markdown using:

### Research Question

### Key Findings

### Source Comparison

### Conclusion

Keep the answer useful and focused on the research question.
"""

In [ ]:
# Build a single research context containing the content and source information.
def build_research_context(sources):
    context = ""

    for index, source in enumerate(sources, start=1):
        context += f"""
SOURCE {index}
URL: {source['url']}

{source['content']}

--------------------
"""

    return context

In [ ]:
# Send the research question and collected sources to the LLM and return the synthesized research.
def research(question, sources):
    context = build_research_context(sources)

    messages = [
        {"role": "system", "content": research_prompt},
        {
            "role": "user",
            "content": f"""
Research Question:

{question}

Research Sources:

{context}
"""
        }
    ]

    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages
    )

    return response.choices[0].message.content

In [ ]:
# Run the research assistant and display the synthesized research findings.
result = research(research_question, sources)

print(result)

In [ ]:
# Define simple evaluation criteria for checking whether important research topics appear in the result.
expected_topics = [
    "customer service",
    "marketing",
    "operations",
    "productivity"
]

result_lower = result.lower()

found_topics = []

for topic in expected_topics:
    if topic in result_lower:
        found_topics.append(topic)

coverage = len(found_topics) / len(expected_topics)

print(f"Topics found: {found_topics}")
print(f"Research topic coverage: {coverage:.2%}")

In [ ]:
# Display the sources used by the research assistant so the findings can be traced back to their origins.
for index, source in enumerate(sources, start=1):
    print(f"Source {index}: {source['url']}")